# FMA Small: Audio-Only Unsupervised Analysis

This notebook uses **only the raw audio** from the FMA Small dataset (8,000 x 30 s MP3 clips). It deliberately **ignores every metadata file** (`tracks.csv`, `genres.csv`, `features.csv`, `echonest.csv`, ...). The only thing taken from a filename is the track ID, which is used as an identifier.

**Pipeline**
1. Discover audio files under `/kaggle/input`
2. Inspect a few waveforms / spectrograms
3. Extract handcrafted features (MFCC, chroma, spectral, rhythm) in parallel
4. PCA, UMAP/t-SNE and KMeans clustering
5. Interpret clusters by listening and by feature profile
6. Content-based similarity search ("find tracks that sound like this one")
7. *(Optional, GPU recommended)* Convolutional autoencoder on log-mel spectrograms to learn embeddings directly from audio

**Setup:** in the Kaggle sidebar choose *Add Input* and search for an FMA small dataset (e.g. "fma small"). Any version containing `fma_small/` MP3s works, because the notebook searches recursively. Internet access is not required.

In [ ]:
import os, gc, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from IPython.display import Audio, display

warnings.filterwarnings("ignore")

# ---------------- Configuration ----------------
SEED        = 42
INPUT_DIR   = Path(os.environ.get("FMA_DIR", "/kaggle/input"))
WORK_DIR    = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
WORK_DIR.mkdir(exist_ok=True, parents=True)
AUDIO_EXTS  = {".mp3", ".wav", ".flac"}
MAX_FILES   = None      # set e.g. 500 for a quick dry run, None for everything
SR          = 22050     # sample rate used for analysis
DURATION    = 30.0      # seconds of audio per clip
N_JOBS      = -1        # use all CPU cores
USE_CACHE   = True      # reuse saved features if present

random.seed(SEED); np.random.seed(SEED)
print("librosa", librosa.__version__)

## 1. Discover audio files (no metadata used)

In [ ]:
files = sorted(p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in AUDIO_EXTS)
if MAX_FILES:
    random.Random(SEED).shuffle(files)
    files = sorted(files[:MAX_FILES])

print(f"Found {len(files):,} audio files")
assert files, "No audio found. Add an FMA small dataset via 'Add Input' in the sidebar."
print("Example path:", files[0])

# Track ID = filename stem (e.g. 000002.mp3 -> 2). It is used purely as an identifier.
track_ids = [int(p.stem) if p.stem.isdigit() else p.stem for p in files]

## 2. Look at a few clips

In [ ]:
sample_paths = random.Random(SEED).sample(files, 3)

fig, axes = plt.subplots(len(sample_paths), 2, figsize=(14, 3.2 * len(sample_paths)))
for row, p in zip(axes, sample_paths):
    y, sr = librosa.load(p, sr=SR, duration=DURATION)
    librosa.display.waveshow(y, sr=sr, ax=row[0], alpha=0.7)
    row[0].set_title(f"Waveform: {p.name}  ({len(y)/sr:.1f}s)")
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128), ref=np.max)
    img = librosa.display.specshow(S, sr=sr, x_axis="time", y_axis="mel", ax=row[1])
    row[1].set_title("Log-mel spectrogram")
plt.tight_layout(); plt.show()

for p in sample_paths:
    print(p.name); display(Audio(filename=str(p)))

## 3. Handcrafted audio feature extraction

For each clip we compute frame-level descriptors and summarise each over time with **mean and standard deviation**:

| Group | Descriptors |
|---|---|
| Timbre | 20 MFCCs and their deltas |
| Harmony | 12 chroma bins |
| Spectral shape | centroid, bandwidth, rolloff, flatness, 7-band contrast |
| Energy / noisiness | RMS, zero-crossing rate |
| Rhythm | estimated tempo, onset strength |

Corrupt or too-short files are skipped rather than crashing the run. A few clips in FMA are known to be unreadable.

In [ ]:
def _ms(name, arr, out):
    """Add mean/std of each row of a (n_features, n_frames) array to dict `out`."""
    arr = np.atleast_2d(arr)
    for i in range(arr.shape[0]):
        out[f"{name}{i+1:02d}_mean"] = float(arr[i].mean())
        out[f"{name}{i+1:02d}_std"]  = float(arr[i].std())

def extract_features(path, sr=SR, duration=DURATION):
    try:
        y, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
        if len(y) < sr * 5:            # skip near-empty clips
            return None
        f = {}
        stft  = np.abs(librosa.stft(y, n_fft=2048, hop_length=512))
        power = stft ** 2
        mel   = librosa.feature.melspectrogram(S=power, sr=sr, n_mels=128)
        mfcc  = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=20)

        _ms("mfcc",        mfcc, f)
        _ms("mfcc_delta",  librosa.feature.delta(mfcc), f)
        _ms("chroma",      librosa.feature.chroma_stft(S=power, sr=sr), f)
        _ms("contrast",    librosa.feature.spectral_contrast(S=stft, sr=sr), f)
        _ms("centroid",    librosa.feature.spectral_centroid(S=stft, sr=sr), f)
        _ms("bandwidth",   librosa.feature.spectral_bandwidth(S=stft, sr=sr), f)
        _ms("rolloff",     librosa.feature.spectral_rolloff(S=stft, sr=sr), f)
        _ms("flatness",    librosa.feature.spectral_flatness(S=stft), f)
        _ms("rms",         librosa.feature.rms(S=stft), f)
        _ms("zcr",         librosa.feature.zero_crossing_rate(y, hop_length=512), f)

        onset = librosa.onset.onset_strength(y=y, sr=sr)
        tempo, _ = librosa.beat.beat_track(onset_envelope=onset, sr=sr)
        f["tempo"]        = float(np.atleast_1d(tempo)[0])
        f["onset_mean"]   = float(onset.mean())
        f["onset_std"]    = float(onset.std())
        return f
    except Exception as e:
        return None

# quick sanity check on one file
test = extract_features(files[0])
print(f"{len(test)} features per track" if test else "First file failed to decode")

In [ ]:
FEAT_PATH = WORK_DIR / "audio_features.csv"

if USE_CACHE and FEAT_PATH.exists():
    feats = pd.read_csv(FEAT_PATH, index_col="track_id")
    print("Loaded cached features:", feats.shape)
else:
    gen = Parallel(n_jobs=N_JOBS, return_as="generator")(delayed(extract_features)(p) for p in files)
    rows, ids, paths, failed = [], [], [], []
    for tid, p, r in zip(track_ids, files, tqdm(gen, total=len(files), desc="Extracting")):
        if r is None: failed.append(p.name)
        else: rows.append(r); ids.append(tid); paths.append(str(p))
    feats = pd.DataFrame(rows, index=pd.Index(ids, name="track_id"))
    feats["path"] = paths
    feats.to_csv(FEAT_PATH)
    print(f"Done: {len(feats):,} ok, {len(failed)} skipped {failed[:5]}")

path_col = feats.pop("path")
X_raw = feats.replace([np.inf, -np.inf], np.nan).fillna(feats.median())
print("Feature matrix:", X_raw.shape)

## 4. Scale, reduce, visualise

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = StandardScaler().fit_transform(X_raw.values)

pca_full = PCA().fit(X)
cum = np.cumsum(pca_full.explained_variance_ratio_)
n_keep = int(np.searchsorted(cum, 0.95) + 1)
plt.figure(figsize=(6, 3.5)); plt.plot(cum); plt.axhline(0.95, ls="--", c="gray")
plt.xlabel("Components"); plt.ylabel("Cumulative variance"); plt.title(f"{n_keep} PCs explain 95%")
plt.show()

Z = PCA(n_components=n_keep, random_state=SEED).fit_transform(X)
print("PCA space:", Z.shape)

In [ ]:
def embed_2d(Z, seed=SEED):
    try:
        import umap
        return umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=seed).fit_transform(Z), "UMAP"
    except Exception:
        from sklearn.manifold import TSNE
        return TSNE(perplexity=30, random_state=seed, init="pca").fit_transform(Z), "t-SNE"

xy, method = embed_2d(Z)
plt.figure(figsize=(7, 6)); plt.scatter(xy[:, 0], xy[:, 1], s=3, alpha=0.4)
plt.title(f"{method} of handcrafted audio features"); plt.axis("off"); plt.show()

## 5. Cluster the audio (KMeans)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks, inertias, sils = list(range(2, 15)), [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=5, random_state=SEED).fit(Z)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Z, km.labels_, sample_size=min(3000, len(Z)), random_state=SEED))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ks, inertias, "o-"); ax[0].set_title("Elbow (inertia)"); ax[0].set_xlabel("k")
ax[1].plot(ks, sils, "o-");     ax[1].set_title("Silhouette");      ax[1].set_xlabel("k")
plt.tight_layout(); plt.show()
print("Best silhouette at k =", ks[int(np.argmax(sils))])

In [ ]:
K = 8   # change freely; silhouette is often low for music, so choose by usefulness as well

km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(Z)
labels = km.labels_
print(pd.Series(labels).value_counts().sort_index().to_string())

plt.figure(figsize=(7, 6))
sc = plt.scatter(xy[:, 0], xy[:, 1], c=labels, cmap="tab10", s=3, alpha=0.6)
plt.legend(*sc.legend_elements(), title="cluster", loc="best", markerscale=1.5)
plt.title(f"KMeans (k={K}) on {method} map"); plt.axis("off"); plt.show()

## 6. What does each cluster sound like?

In [ ]:
profile_cols = ["tempo", "rms_01_mean", "centroid01_mean", "bandwidth01_mean", "rolloff01_mean",
                "flatness01_mean", "zcr01_mean", "onset_mean", "contrast03_mean", "contrast06_mean",
                "mfcc02_mean", "mfcc03_mean"]
profile_cols = [c for c in profile_cols if c in X_raw.columns]

prof = X_raw[profile_cols].groupby(labels).mean()
prof_z = (prof - prof.mean()) / prof.std()

plt.figure(figsize=(10, 4))
plt.imshow(prof_z.T, aspect="auto", cmap="coolwarm", vmin=-2, vmax=2)
plt.yticks(range(len(profile_cols)), profile_cols); plt.xticks(range(K), [f"C{i}" for i in range(K)])
plt.colorbar(label="z-score across clusters"); plt.title("Cluster feature profiles"); plt.show()

In [ ]:
# Listen to the 2 tracks nearest each cluster centre
paths_arr = path_col.values
for c in range(K):
    idx = np.where(labels == c)[0]
    d = np.linalg.norm(Z[idx] - km.cluster_centers_[c], axis=1)
    for j in idx[np.argsort(d)[:2]]:
        print(f"Cluster {c} | track {X_raw.index[j]}")
        display(Audio(filename=paths_arr[j]))

## 7. Content-based similarity search

Given any track, find the clips that sound closest to it using cosine distance in PCA feature space.

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=6, metric="cosine").fit(Z)

def similar_to(track_id, n=5):
    i = X_raw.index.get_loc(track_id)
    dist, ind = nn.kneighbors(Z[i:i+1], n_neighbors=n + 1)
    print(f"QUERY: track {track_id}"); display(Audio(filename=paths_arr[i]))
    for d, j in zip(dist[0][1:], ind[0][1:]):
        print(f"  neighbour {X_raw.index[j]}  (cosine dist {d:.3f})")
        display(Audio(filename=paths_arr[j]))

similar_to(int(random.Random(1).choice(list(X_raw.index))))

## 8. (Optional) Learn embeddings with a convolutional autoencoder

Instead of hand-picked features, this trains a small CNN autoencoder on **log-mel spectrogram patches** (128 mel bands x 128 frames, about 6 s from the middle of each clip). The bottleneck vector becomes a learned audio embedding. Turn on a GPU (*Settings -> Accelerator*) for this section.

In [ ]:
N_MELS, N_FRAMES, HOP = 128, 128, 1024

def mel_patch(path, sr=SR):
    try:
        y, _ = librosa.load(path, sr=sr, duration=DURATION, mono=True)
        S = librosa.power_to_db(librosa.feature.melspectrogram(
                y=y, sr=sr, n_fft=2048, hop_length=HOP, n_mels=N_MELS), ref=np.max)
        if S.shape[1] < N_FRAMES: return None
        s = (S.shape[1] - N_FRAMES) // 2
        return S[:, s:s + N_FRAMES].astype(np.float16)
    except Exception:
        return None

MEL_PATH = WORK_DIR / "mel_patches.npz"
if USE_CACHE and MEL_PATH.exists():
    d = np.load(MEL_PATH); mels, mel_ids = d["mels"], d["ids"]
else:
    gen = Parallel(n_jobs=N_JOBS, return_as="generator")(delayed(mel_patch)(p) for p in paths_arr)
    out = [(tid, m) for tid, m in zip(X_raw.index, tqdm(gen, total=len(paths_arr), desc="Mel patches")) if m is not None]
    mel_ids = np.array([t for t, _ in out]); mels = np.stack([m for _, m in out])
    np.savez_compressed(MEL_PATH, mels=mels, ids=mel_ids)
print(mels.shape, mels.dtype)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
print("Device:", dev)

mu, sd = float(mels.astype(np.float32).mean()), float(mels.astype(np.float32).std())
tens = torch.from_numpy(((mels.astype(np.float32) - mu) / sd)).unsqueeze(1)    # (N,1,128,128)
loader = DataLoader(TensorDataset(tens), batch_size=64, shuffle=True)

def block(i, o):  return nn.Sequential(nn.Conv2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.LeakyReLU(0.2))
def dblock(i, o): return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU())

class AE(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.enc  = nn.Sequential(block(1, 32), block(32, 64), block(64, 128), block(128, 256))   # 128->8
        self.to_z = nn.Linear(256 * 8 * 8, latent)
        self.from_z = nn.Linear(latent, 256 * 8 * 8)
        self.dec  = nn.Sequential(dblock(256, 128), dblock(128, 64), dblock(64, 32),
                                  nn.ConvTranspose2d(32, 1, 4, 2, 1))
    def encode(self, x): return self.to_z(self.enc(x).flatten(1))
    def forward(self, x):
        z = self.encode(x)
        return self.dec(self.from_z(z).view(-1, 256, 8, 8)), z

model = AE().to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS, losses = 15, []
for ep in range(EPOCHS):
    model.train(); tot = 0
    for (xb,) in loader:
        xb = xb.to(dev); recon, _ = model(xb)
        loss = nn.functional.mse_loss(recon, xb)
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(xb)
    losses.append(tot / len(tens)); print(f"epoch {ep+1:2d}  mse {losses[-1]:.4f}")

plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.title("Autoencoder training"); plt.show()

In [ ]:
# Extract embeddings, then cluster and visualise exactly as before
model.eval(); embs = []
with torch.no_grad():
    for i in range(0, len(tens), 256):
        embs.append(model.encode(tens[i:i+256].to(dev)).cpu().numpy())
E = StandardScaler().fit_transform(np.concatenate(embs))

xy_ae, m2 = embed_2d(E)
km_ae = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(E)
print("Silhouette  handcrafted:", round(silhouette_score(Z, labels, sample_size=3000, random_state=SEED), 3),
      "| autoencoder:", round(silhouette_score(E, km_ae.labels_, sample_size=3000, random_state=SEED), 3))

plt.figure(figsize=(7, 6))
plt.scatter(xy_ae[:, 0], xy_ae[:, 1], c=km_ae.labels_, cmap="tab10", s=3, alpha=0.6)
plt.title(f"Autoencoder embeddings ({m2}), KMeans k={K}"); plt.axis("off"); plt.show()

# Save embeddings for reuse (indexed by track ID only)
pd.DataFrame(E, index=pd.Index(mel_ids, name="track_id")).to_csv(WORK_DIR / "autoencoder_embeddings.csv")

## Notes and next steps

* **No labels are used.** Quality is judged by listening, cluster profiles and silhouette score. Silhouette values are usually low for music because genres overlap.
* **Outputs** in `/kaggle/working`: `audio_features.csv`, `mel_patches.npz`, `autoencoder_embeddings.csv`.
* **Ideas to extend:** contrastive/self-supervised training with augmentations (pitch shift, time stretch, noise), pretrained audio encoders (e.g. PANNs, VGGish), HDBSCAN instead of KMeans, or a playlist generator driven by the similarity search.
* **Runtime:** feature extraction for all 8,000 clips takes roughly 15-30 min on a 4-core Kaggle CPU. Set `MAX_FILES = 500` for a quick dry run.